# Phase 2 — PyTorch Star Center Detection (`StarCenterUNetMini`)

Thin orchestration notebook calling reusable modules in `src/constellation_detection/`.
Compatible with **Google Colab GPU** (`requirements-colab.txt`) and **Local CPU** (`venv/`).

In [ ]:
# 0. Optional Colab Bootstrap (no hardcoded Drive mounts or personal paths)
import os
if os.environ.get('COLAB_RELEASE_TAG'):
    print('Detected Google Colab runtime. Ensure repo is checked out and run:')
    print('  %pip install -r requirements-colab.txt')

In [ ]:
# 1-3. Environment inspection, dependency verification & runtime device probe
import json
from torch.utils.data import DataLoader
from constellation_detection.artifacts import RunArtifactManager, build_run_id
from constellation_detection.config import ProjectPaths
from constellation_detection.datasets import StarHeatmapDataset, audit_pilot_dataset, load_annotation_record, load_split_manifest
from constellation_detection.detection.dl import build_model, detect_stars_dl
from constellation_detection.evaluation.matching import compute_average_precision, evaluate_phase1_baselines_on_manifest, match_detections_to_ground_truth, select_best_threshold
from constellation_detection.image.io import load_image
from constellation_detection.runtime import collect_env_snapshot, make_dataloader_worker_init, resolve_device, seed_everything
from constellation_detection.training import StarCenterLoss, TrainingConfig, run_training

env_snap = collect_env_snapshot()
print(json.dumps(env_snap, indent=2))
device, dev_info = resolve_device(preferred='auto', require_gpu=False)
print('Resolved device:', dev_info)

In [ ]:
# 4-8. Dataset loading, validation, annotation inspection, visualization & split check
import matplotlib.pyplot as plt
import numpy as np

paths = ProjectPaths.from_paths()
manifest_path = paths.splits_dir / 'dataset_manifest_v1.0.json'
manifest = load_split_manifest(manifest_path)
audit_report = audit_pilot_dataset(manifest_path, paths.data_dir)
print('Dataset audit verdict:', audit_report['verdict'], '| split_counts:', audit_report['split_counts'])
assert audit_report['verdict'] != 'FAIL', audit_report['issues']

cfg = TrainingConfig()
seed_everything(cfg.seed, deterministic=cfg.deterministic)
train_ds = StarHeatmapDataset(manifest_path, paths.data_dir, split='train', crop_size=cfg.crop_size, augment=True, sigma_px=cfg.sigma_px, seed=cfg.seed)
val_ds = StarHeatmapDataset(manifest_path, paths.data_dir, split='val', crop_size=cfg.crop_size, augment=False, sigma_px=cfg.sigma_px, seed=cfg.seed)
print('Train crops:', len(train_ds), '| Val crops:', len(val_ds))

sample_item = train_ds[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample_item['image'][0].numpy(), cmap='gray')
axes[0].set_title(f'Train crop: {sample_item["image_id"]}')
axes[1].imshow(sample_item['heatmap'][0].numpy(), cmap='magma')
axes[1].set_title('Rendered Gaussian Star Target')
plt.tight_layout()
plt.show()

In [ ]:
# 9-14. Model construction, loss check, optimizer/scheduler, training, validation & checkpoints
RUN_TRAINING = False  # Explicitly set True only on approved GPU training runs

model = build_model(cfg.model_name, {'in_channels': cfg.in_channels, 'base_channels': cfg.base_channels})
print('Parameters:', sum(p.numel() for p in model.parameters()))
criterion = StarCenterLoss(alpha=cfg.focal_alpha, beta=cfg.focal_beta, lambda_offset=cfg.lambda_offset)
worker_fn = make_dataloader_worker_init(cfg.seed)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, worker_init_fn=worker_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

run_id = build_run_id('p2', 'stardet', 'unetmini', cfg.dataset_version, cfg.seed)
artifact_mgr = RunArtifactManager(paths.runs_dir, run_id)
artifact_mgr.write_json_atomic('run_metadata.json', {'run_id': run_id, 'env': env_snap, 'device': dev_info.to_dict()})

if RUN_TRAINING:
    summary = run_training(model, train_loader, val_loader, cfg, artifact_mgr, dev_info)
    print('Training completed:', summary['best_epoch'], summary['best_val_loss'])
else:
    print('Dry-run verified cleanly. Set RUN_TRAINING = True to start GPU training.')

In [ ]:
# 15-17. Frozen Phase 1 baseline benchmark, validation-selected threshold & transfer bundle
baseline_rows = evaluate_phase1_baselines_on_manifest(manifest_path, paths.data_dir, split='test', tolerance_px=3.0)
for row in baseline_rows:
    print(row)

# When RUN_TRAINING is True, sweep threshold on 'val' and export bundle for local Windows CPU verification:
# val_samples = [s for s in manifest['samples'] if s['split'] == 'val']
# best_t, val_summary = select_best_threshold(val_preds, val_gt, tolerance_px=3.0)
# bundle_path = artifact_mgr.export_run_bundle(include_trainstate=False)